# Hybrid ALNS Performance Evaluation

---

## I. Purpose

This notebook defines a **controlled benchmark procedure** to evaluate the Hybrid ALNS solver,
which utilizes an offline repair model to guide neighborhood reconstruction.

## II. Experimental Preconditions and Reproducibility Controls

Required artifacts:
- trained model `bin_packing/hybrid_ml_metaheuristics/hybrid_alns/models/repair_model_v2.pkl`
- benchmark instance directory (`datasets/` at the repo root)
- fixed random seed and defined iteration / time budgets

In [ ]:
from bin_packing_optimization.utilities import Benchmark, create_benchmark
import pickle
from importlib.resources import files
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
    solver as hybrid_alns_solver,
)

## III. Hybrid Run (ALNS with Offline Repair Model)

Runs the full benchmark via the `Benchmark` class — no CLI shell-out.

In [ ]:
model_resource = (
    files("bin_packing.hybrid_ml_metaheuristics.hybrid_alns.models")
    / "repair_model_v2.pkl"
)
with model_resource.open("rb") as f:
    model_bundle = pickle.load(f)

benchmark: Benchmark = create_benchmark(
    dataset_key="falkenauer-t", solver_module=hybrid_alns_solver
)
benchmark.run(
    method=None,
    method_args={
        "max_iterations": 2000,
        "model_bundle": model_bundle,
    },
)

results = benchmark.get_results()
if results:
    csv_path = benchmark.save_results_to_csv()
    print("Results saved to:", csv_path)

## IV. Summary Statistics

In [ ]:
from bin_packing_optimization.utilities.statistics import (
    summarize,
    summarize_by_size,
    ResultRow,
)

rows = [
    ResultRow(
        instance_name=r.instance_name,
        dataset_key=r.dataset_key,
        num_items=r.num_items,
        bin_capacity=r.bin_capacity,
        bins_used=r.bins_used,
        lower_bound=r.lower_bound,
        total_weight=r.total_weight,
        elapsed_time=r.elapsed_time,
        method=r.method,
        timed_out=r.timed_out,
    )
    for r in results
]

summary = summarize(rows)
by_size = summarize_by_size(rows)

print("\n=== Overall ===")
for k, v in summary.items():
    print(f"  {k:<30s}: {v:.4f}" if isinstance(v, float) else f"  {k:<30s}: {v}")

print("\n=== By instance size ===")
for s in by_size:
    print(
        f"  n={s.num_items:4d} | completed={s.completed}/{s.instances}"
        f" | avg_time={s.avg_time_s:.4f}s | avg_gap={s.avg_gap:.3f}"
    )

## V. Graphs

In [ ]:
from bin_packing_optimization.utilities.graphing import create_graphs

graph_paths = create_graphs(csv_path)
print("Graphs written:")
for p in graph_paths:
    print(" ", p)